# Will It Go Viral?
### A Logistic Regression model, trained on **real YouTube Shorts**, that estimates whether a short video will go viral from its **hook**, **structure** and **content**

**Author:** Gustavo Pugliezi · **Course:** Computer Science · **Tools:** Python, yt-dlp, OpenCV, pandas, scikit-learn, matplotlib

---

## Contents
| Step | Framework stage | What happens |
|---|---|---|
| [1](#s1) | Define the problem and label | what "viral" means, as a number |
| [2](#s2) | Define measurable features | 11 characteristics with objective measuring rules |
| [3](#s3) | Receive posts as JSON | real Shorts collected from YouTube |
| [4](#s4) | Extract feature values | an LLM, or a classroom rule set, reads each post |
| [5](#s5) | Build the dataset | one row per video, one column per feature |
| [6](#s6) | Separate `X` and `y` | features vs. label |
| [7](#s7) | Train/test split | 80% to learn, 20% kept hidden for the final exam |
| [8](#s8) | Train a Logistic Regression | the model learns one weight per feature |
| [9](#s9) | Evaluate | accuracy and confusion matrix |
| [10](#s10) | Test with brand-new data | recent videos whose outcome is not known yet |
| [11](#s11) | Export as `.pkl` | save the trained model to a file |
| [12](#s12) | Open and run in any application | a separate program loads the `.pkl` and predicts |

## How this notebook is explained
I learned machine learning through one analogy, and I kept it here so the reasoning behind every step is visible.

> **The intern analogy.** Imagine hiring an intern who knows *nothing* about videos. You cannot hand them a rulebook, because nobody knows the exact rules of virality. Instead, you show them hundreds of real videos together with what actually happened, and they gradually work out the rules on their own. Then you test them on videos they have never seen.

Every step has two layers:
- **Intern analogy**: the intuition, in plain language.
- **Technical note**: what the code actually does.

## How to run
1. Run **`collect_real_videos.ipynb`** first, on your own computer. It collects real Shorts from YouTube and saves `posts.json` and `new_posts.json` in this folder.
2. Open this notebook in the same folder and choose **Run → Run All Cells**.

Every result in this notebook (charts, weights, accuracy) is computed from the collected data, so the numbers change if the data is collected again.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib

In [ ]:
import os, sys, json, pickle, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Bins whose width are too small.*")

SEED = 42  # fixes every random split, so the notebook gives the same results every time

INK, PAPER, RED, GREY = "#15171c", "#f8f9f6", "#e8341c", "#8a8f96"
plt.rcParams.update({"figure.facecolor": PAPER, "axes.facecolor": PAPER, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold", "figure.dpi": 110})

print(f"Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | pandas {pd.__version__}")
print("Working folder:", os.getcwd())

<a id="s1"></a>
## Step 1 — Define the problem and the label

> **Intern analogy.** Before training anyone, you agree on what *success* means. "Viral" cannot be a feeling; it must be something you can check. Otherwise the intern is learning to hit a target that keeps moving.

**Problem:** a creator wants to know, *before posting*, whether a Short is likely to go viral.

**Type of problem:** supervised binary classification. *Supervised* because every training example comes with the right answer; *binary* because there are only two answers.

**Label (`y`).** Big channels get many views simply because they are big, so a video is judged against its own channel's size, using the **views per subscriber** ratio:

$$
\text{ratio} = \frac{\text{views}}{\text{subscribers}} \qquad\qquad \text{viral} = \begin{cases} 1 & \text{if the ratio is in the top 25\% of the collected videos} \\ 0 & \text{otherwise} \end{cases}
$$

In words: a video is *viral* when it reached far more people than its channel size would suggest, compared with the other real Shorts in the sample. The exact cut-off (for example, "at least 2.7 views per subscriber") is computed from the data in Step 5 and then fixed.

In [ ]:
VIRAL_SHARE = 0.25   # top 25% of views-per-subscriber count as viral

def views_per_subscriber(post):
    return post["views"] / post["followers"]

<a id="s2"></a>
## Step 2 — Define measurable features

> **Intern analogy.** The intern cannot "watch" a video intelligently, so you give them a **checklist** with 11 questions to fill in for every video. Each question must be objective: two different people checking the same video must write down the same answer.

All features describe things the creator **controls before posting**. The channel's size is deliberately *not* a feature, because it is part of the label.

| Group | Feature | Type | Measuring rule | Measured from |
|---|---|---|---|---|
| **Hook** | `first_words_s` | number | second at which the first words are spoken (no speech → 10) | captions |
| | `words_first_5s` | number | words spoken in the first 5 seconds | captions |
| | `hook_type` | category | question, shock, promise, story or none, judged from the first ~5 seconds | captions |
| | `face_first_second` | 0/1 | a face is detected in the first second | video frames |
| **Structure** | `duration_s` | number | length in seconds | metadata |
| | `cuts_per_10s` | number | (number of cuts ÷ duration) × 10 | video frames |
| | `has_loop` | 0/1 | the last frame looks like the first one | video frames |
| | `structure` | category | list, tutorial, before_after, storytime or other | title + description |
| **Content** | `topic` | category | humor, education, beauty, gaming, cooking or sports | search topic |
| | `title_question` | 0/1 | the title contains a question mark | title |
| | `posted_weekend` | 0/1 | posted on Saturday or Sunday | upload date |

In [ ]:
NUMERIC = ["first_words_s", "words_first_5s", "duration_s", "cuts_per_10s"]
BINARY = ["face_first_second", "has_loop", "title_question", "posted_weekend"]
CATEGORICAL = ["hook_type", "structure", "topic"]
FEATURES = NUMERIC + BINARY + CATEGORICAL
print(f"{len(FEATURES)} features:", FEATURES)

<a id="s3"></a>
## Step 3 — Receive the posts as JSON

> **Intern analogy.** You hand the intern a pile of real videos. They are not video files; they are written records: the title, what was said and when (from YouTube's automatic captions), and a few measurements taken from the frames.

**Technical note.** The posts were collected by `collect_real_videos.ipynb`:
- searched YouTube Shorts in Portuguese for 6 topics;
- kept only Shorts of up to 90 s, posted **between 7 and 365 days before collection** (Shorts get most of their views in the first week, so younger videos are not "finished" yet);
- skipped channels that hide their subscriber count;
- downloaded each video once to measure faces, cuts and loops, then deleted it.

`views`, `followers` and `likes` exist **only** to compute the label.

In [ ]:
if not os.path.exists("posts.json"):
    raise FileNotFoundError("posts.json not found: run collect_real_videos.ipynb first, in this same folder.")

with open("posts.json", encoding="utf-8") as f:
    posts = json.load(f)

channels = {p.get("channel") for p in posts}
dates = sorted(p["posted_date"] for p in posts)
print(f"{len(posts)} real Shorts from {len(channels)} channels, posted between {dates[0]} and {dates[-1]}")
print(f"Collected on: {sorted({p['collected_at'] for p in posts})}")
print("Videos per topic:", pd.Series([p["topic"] for p in posts]).value_counts().to_dict(), "\n")

example = {k: v for k, v in posts[0].items() if k != "description"}
example["transcript"] = example["transcript"][:300] + ("…" if len(example["transcript"]) > 300 else "")
print(json.dumps(example, ensure_ascii=False, indent=2))

<a id="s4"></a>
## Step 4 — Extract the feature values

> **Intern analogy.** Someone has to read each post and fill in the checklist. Most answers are simple measurements (when the first word is spoken, how long the video is). Two answers need *understanding language*: what kind of hook the video opens with, and what format it has. For those we have two helpers: an AI that reads text (an **LLM**), or, for the classroom, a set of key-phrase rules (e.g. *"aprenda"* or *"segredo"* at the start means a *promise* hook). One rule is non-negotiable: whoever fills in the checklist must **never see the views, likes or subscribers**. That would be like giving the intern the answer key together with the exam.

| | 4a. LLM | 4b. Classroom rules |
|---|---|---|
| Used for | `hook_type` and `structure` | `hook_type` and `structure` |
| Needs | internet + `ANTHROPIC_API_KEY` | nothing |
| Strength | understands free-form speech | fast, free, reproducible |

**Technical note.** The code is written to `post_extractor.py`, so the app in Step 12 reuses exactly the same extraction. Everything passes through `validate_features`, which converts types and **refuses** values outside the allowed ones instead of silently guessing.

In [ ]:
%%writefile post_extractor.py
"""Feature extraction for the "Will It Go Viral?" project (Gustavo Pugliezi)."""
import json
import re
from datetime import datetime

NO_SPEECH_S = 10.0
ALLOWED = {
    "hook_type": {"question", "shock", "promise", "story", "none"},
    "structure": {"list", "tutorial", "before_after", "storytime", "other"},
    "topic": {"humor", "education", "beauty", "gaming", "cooking", "sports"},
}
NUMERIC = ["first_words_s", "words_first_5s", "duration_s", "cuts_per_10s"]
BINARY = ["face_first_second", "has_loop", "title_question", "posted_weekend"]
FEATURES = NUMERIC + BINARY + list(ALLOWED)
# What an extractor may read. views, likes and followers are left out on purpose: they define the label.
VISIBLE_FIELDS = ["id", "title", "description", "transcript", "duration_s", "posted_date", "topic"]


def validate_features(raw):
    """Converts types and checks every value. Raises ValueError listing all problems."""
    try:
        clean = {k: float(raw[k]) for k in NUMERIC}
        clean.update({k: int(raw[k]) for k in BINARY})
        clean.update({k: str(raw[k]).strip().lower() for k in ALLOWED})
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError(f"missing or malformed feature: {error}")
    problems = [f"{k}={clean[k]!r} not in {sorted(v)}" for k, v in ALLOWED.items() if clean[k] not in v]
    problems += [f"{k} must be 0 or 1" for k in BINARY if clean[k] not in (0, 1)]
    problems += [f"{k} is negative" for k in NUMERIC if clean[k] < 0]
    if not 0 < clean["duration_s"] <= 600:
        problems.append("duration_s out of range")
    if problems:
        raise ValueError("; ".join(problems))
    return {k: clean[k] for k in FEATURES}


def parse_transcript(transcript):
    """'[00:01.2] text' lines -> list of (seconds, text)."""
    lines = []
    for line in transcript.split("\n"):
        m = re.match(r"\[(\d+):(\d+(?:\.\d+)?)\]\s*(.*)", line.strip())
        if m and m.group(3).strip():
            lines.append((int(m.group(1)) * 60 + float(m.group(2)), m.group(3).strip()))
    return lines


# ---------------------------------------------------------------- 4b. classroom rules
HOOK_RULES = [  # order matters: the first match wins
    ("promise", r"vou te (mostrar|ensinar)|aprenda|segredo|truque|dica|jeito mais|passo a passo|"
                r"i'll show|secret|trick|hack|learn"),
    ("question", r"\?|você sabia|sabia que|por que|porque será|qual (é|o|a)|quem (é|foi)|"
                 r"did you know|what|why|how come"),
    ("shock", r"não acredit|olha (isso|o que|só)|para tudo|nunca vi|absurd|chocad|incrível|insano|"
              r"can't believe|look at this|insane|craziest"),
    ("story", r"aconteceu|um dia|quando eu|ontem|história|me contou|storytime|happened|one day|when i"),
]
STRUCTURE_RULES = [
    ("list", r"\btop ?\d+|\b\d+ (coisas|dicas|formas|jeitos|motivos|things|tips|ways)"),
    ("before_after", r"antes e depois|antes x depois|before and after|transforma"),
    ("storytime", r"storytime|história|aconteceu|story"),
    ("tutorial", r"tutorial|como fazer|passo a passo|receita|aprenda|how to|recipe"),
]


def first_match(rules, text, default):
    for name, regex in rules:
        if re.search(regex, text, re.IGNORECASE):
            return name
    return default


def classify_with_rules(post, opening):
    hook_type = first_match(HOOK_RULES, opening, "none") if opening else "none"
    structure = first_match(STRUCTURE_RULES, post["title"] + " " + post.get("description", ""), "other")
    return hook_type, structure


# ---------------------------------------------------------------- 4a. LLM
def classify_with_llm(post, opening, model="claude-haiku-5-5"):
    import anthropic
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    visible = {k: post[k] for k in VISIBLE_FIELDS if k in post}
    prompt = (
        "You annotate short-form videos. Using only the post below, answer with ONLY a JSON object "
        'with two keys: "hook_type" (one of: question, shock, promise, story, none; judge only the '
        "speech in the first ~5 seconds, quoted under OPENING) and \"structure\" (one of: list, tutorial, "
        "before_after, storytime, other; judge from the title and description).\n\n"
        f"OPENING: {opening!r}\n\nPOST:\n{json.dumps(visible, ensure_ascii=False, indent=1)}"
    )
    response = client.messages.create(model=model, max_tokens=100,
                                      messages=[{"role": "user", "content": prompt}])
    text = next(block.text for block in response.content if block.type == "text")
    answer = json.loads(text[text.find("{"): text.rfind("}") + 1])
    return answer["hook_type"], answer["structure"]


# ---------------------------------------------------------------- extraction
def extract(post, use_llm=False):
    lines = parse_transcript(post.get("transcript", ""))
    opening = " ".join(text for t, text in lines if t < 5)
    hook_type, structure = (classify_with_llm if use_llm else classify_with_rules)(post, opening)
    analysis = post["video_analysis"]
    duration = float(post["duration_s"])
    return validate_features({
        "first_words_s": lines[0][0] if lines else NO_SPEECH_S,
        "words_first_5s": len(opening.split()),
        "duration_s": duration,
        "cuts_per_10s": round(analysis["cuts"] / duration * 10, 2),
        "face_first_second": analysis["face_first_second"],
        "has_loop": analysis["has_loop"],
        "title_question": int("?" in post["title"]),
        "posted_weekend": int(datetime.strptime(post["posted_date"], "%Y%m%d").weekday() >= 5),
        "hook_type": hook_type,
        "structure": structure,
        "topic": post["topic"],
    })

In [ ]:
sys.path.insert(0, os.getcwd())
import post_extractor as px
importlib.reload(px)   # picks up changes if the cell above is edited and re-run

USE_LLM = False        # set to True to classify hooks and formats with an LLM (requires ANTHROPIC_API_KEY)
if USE_LLM and not os.environ.get("ANTHROPIC_API_KEY"):
    print("No ANTHROPIC_API_KEY found: using the classroom rules instead.")
    USE_LLM = False

print("Hook/format classifier:", "LLM" if USE_LLM else "classroom rules")
print("\nTitle:", posts[0]["title"])
print("Extracted:", px.extract(posts[0], USE_LLM))

<a id="s5"></a>
## Step 5 — Build the dataset

> **Intern analogy.** All the filled-in checklists go into one big spreadsheet: one row per video, one column per question, and a last column saying whether the video actually went viral.

**Technical note.** Any post that fails validation is reported instead of silently dropped or guessed. The viral cut-off from Step 1 is computed here and printed.

In [ ]:
rows, failed = [], []
for post in posts:
    try:
        rows.append({"id": post["id"], **px.extract(post, USE_LLM), "ratio": views_per_subscriber(post)})
    except (ValueError, KeyError, ZeroDivisionError) as error:
        failed.append((post.get("id"), str(error)))

dataset = pd.DataFrame(rows)
VIRAL_CUTOFF = dataset["ratio"].quantile(1 - VIRAL_SHARE)
dataset["viral"] = (dataset["ratio"] >= VIRAL_CUTOFF).astype(int)
dataset.to_csv("dataset.csv", index=False)

print(f"Dataset: {len(dataset)} rows  →  dataset.csv")
print(f"Posts that failed extraction: {len(failed)}", failed[:5])
print(f"Missing values: {int(dataset[FEATURES].isna().sum().sum())}")
print(f"Viral cut-off: at least {VIRAL_CUTOFF:.2f} views per subscriber "
      f"(median video: {dataset['ratio'].median():.2f})")
dataset.head()

<a id="s6"></a>
## Step 6 — Separate `X` (features) and `y` (label)

> **Intern analogy.** You split the spreadsheet in two: the **questions** (the checklist columns) and the **answers** (viral or not). The intern studies by looking at the questions and then checking the answers.

`id` and `ratio` are left out of `X`: a name says nothing about the video, and the ratio *is* the answer.

In [ ]:
X = dataset[FEATURES]
y = dataset["viral"]
print("X:", X.shape, "| y:", y.shape)
print(f"Viral videos: {y.sum()} of {len(y)} ({y.mean():.1%})")

<a id="s7"></a>
## Step 7 — Split into training and testing sets

> **Intern analogy.** Before any studying starts, you lock 20% of the videos in a drawer. The intern studies the rest. If you tested them on the same videos they studied, they could score 100% just by **memorizing**, without learning anything. The drawer is the surprise final exam.

**Technical note.** 80% training / 20% testing. `stratify=y` keeps the same share of viral videos in both groups.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f"Training: {len(X_train)} videos ({y_train.mean():.1%} viral)")
print(f"Testing:  {len(X_test)} videos ({y_test.mean():.1%} viral)")

### 7.1 Exploring the training set (and only the training set)

> **Intern analogy.** Before teaching, you flip through the study pile to see what the data looks like. You do **not** peek inside the exam drawer: anything you learn from the exam would make the final score look better than it really is.

In [ ]:
train = X_train.assign(viral=y_train)
rate = train.viral.mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, col in zip(axes, CATEGORICAL):
    r = train.groupby(col).viral.mean().sort_values()
    ax.barh(r.index, r.values, color=[RED if v == r.max() else GREY for v in r])
    ax.axvline(rate, color=INK, ls="--", lw=1); ax.set_title(col); ax.set_xlabel("viral rate")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, col in zip(axes, NUMERIC):
    groups = pd.qcut(train[col], 4, duplicates="drop")
    r = train.groupby(groups, observed=True).viral.mean()
    ax.plot(range(len(r)), r.values, marker="o", color=RED, lw=2)
    ax.set_xticks(range(len(r))); ax.set_xticklabels([f"{i.left:.3g}–{i.right:.3g}" for i in r.index], rotation=45)
    ax.axhline(rate, color=INK, ls="--", lw=1); ax.set_title(col)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(18, 3.2))
for ax, col in zip(axes, BINARY):
    r = train.groupby(col).viral.mean().reindex([0, 1])
    ax.bar(["no", "yes"], r.values, color=[GREY, RED]); ax.axhline(rate, color=INK, ls="--", lw=1); ax.set_title(col)
plt.tight_layout(); plt.show()
print("Dashed line = average viral rate in the training set. Numeric features are split into 4 equal-sized groups.")

The cell below turns the charts into sentences, so the observations always match the data that was collected.

In [ ]:
print(f"Baseline: {rate:.0%} of training videos are viral, so 'always answer not viral' is right {1 - rate:.0%} of the time.\n")
for col in CATEGORICAL:
    r = train.groupby(col).viral.agg(["mean", "size"])
    r = r[r["size"] >= 10].sort_values("mean")
    if len(r) > 1:
        print(f"{col:15s} best: {r.index[-1]} ({r['mean'].iloc[-1]:.0%}, {r['size'].iloc[-1]} videos)   "
              f"worst: {r.index[0]} ({r['mean'].iloc[0]:.0%}, {r['size'].iloc[0]} videos)")
for col in NUMERIC:
    q = pd.qcut(train[col], 4, duplicates="drop")
    r = train.groupby(q, observed=True).viral.mean()
    shape = "rises" if r.iloc[-1] > r.iloc[0] + .05 else "falls" if r.iloc[-1] < r.iloc[0] - .05 else "stays about flat"
    peak = " (peaks in the middle)" if r.values.argmax() not in (0, len(r) - 1) else ""
    print(f"{col:15s} viral rate {shape} from the lowest to the highest group: {r.iloc[0]:.0%} → {r.iloc[-1]:.0%}{peak}")
for col in BINARY:
    r = train.groupby(col).viral.mean().reindex([0, 1])
    print(f"{col:15s} without: {r[0]:.0%}   with: {r[1]:.0%}")

**Why this matters for Step 8.** A points system can only say "more X = better" or "more X = worse". If a numeric feature *peaks in the middle* or changes in steps, one weight cannot describe it. The "boxes" in Step 8 solve that.

<a id="s8"></a>
## Step 8 — Train a Logistic Regression

> **Intern analogy: how the intern thinks.** The intern uses a **points system**. Each feature adds or removes points. They add everything up and turn the total into a percentage: lots of points means "likely viral", few points means "unlikely". At the start the intern has no idea how many points each feature deserves, so every weight is 0 and they answer "50%" for everything.
>
> **How the intern learns.** They take a video from the study pile, guess, and check the answer. *"I said 20%, but it went viral. It had a face in the first second, so maybe faces deserve more points."* They nudge the points **a little** (never a lot: one video might just have been lucky) and move on to the next video. They go through the whole pile many times, and each pass their mistakes get a bit smaller. When the adjustments stop making a difference, training is done. They only adjust the features that were **present**: if the video had no face, the face was not responsible, so its points stay the same.
>
> **The "boxes" trick.** Instead of one number for duration or editing pace, each video is sorted into **boxes** (shortest quarter, second quarter, …) and the intern learns a separate score for each box. That way they can discover, for example, that the middle box is the best one.

**Technical note.** The model is a `Pipeline`, an assembly line that always applies the same steps in the same order:

| Stage | What it does | Why |
|---|---|---|
| `StandardScaler` on `first_words_s`, `words_first_5s` | rescales to mean 0, std 1 | one weight each, on the same scale |
| `KBinsDiscretizer` on `duration_s`, `cuts_per_10s` | splits each into boxes with the same number of videos, one 0/1 column per box | can capture curves and steps |
| `passthrough` on the 0/1 features | keeps them as they are | already numbers on the same scale |
| `OneHotEncoder` on categories | one 0/1 column per option | models only understand numbers |
| `LogisticRegression` | learns one weight per column | `chance = 1 / (1 + e^-(b + w₁x₁ + … + wₙxₙ))` |

The number of boxes is chosen by cross-validation on the training set (9.2 explains cross-validation). The box edges, the scaler's mean and the categories are all learned **from the training set only**, inside `fit`. Training minimizes **log loss** (confident mistakes are punished much more than hesitant ones) with the L-BFGS algorithm, a refined version of gradient descent: "feel which way is downhill, take a small step, repeat". A built-in penalty on large weights (L2 regularization, `C=1`) keeps the intern from over-reacting to odd examples, which matters with a few hundred real videos.

In [ ]:
def build_pipeline(model, n_boxes=4):
    preprocess = ColumnTransformer([
        ("scale", StandardScaler(), ["first_words_s", "words_first_5s"]),
        ("boxes", KBinsDiscretizer(n_bins=n_boxes, encode="onehot-dense", strategy="quantile"), ["duration_s", "cuts_per_10s"]),
        ("binary", "passthrough", BINARY),
        ("onehot", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("preprocess", preprocess), ("model", model)])

# How many boxes? Decided with 5-fold cross-validation on the TRAINING set (never the test set).
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
options = {}
for n_boxes in [2, 3, 4, 5]:
    candidate = build_pipeline(LogisticRegression(max_iter=1000), n_boxes)
    options[n_boxes] = cross_val_score(candidate, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    print(f"{n_boxes} boxes → cross-validated AUC {options[n_boxes]:.3f}")
N_BOXES = max(options, key=options.get)
print(f"Chosen: {N_BOXES} boxes\n")

pipeline = build_pipeline(LogisticRegression(max_iter=1000), N_BOXES)
pipeline.fit(X_train, y_train)          # ← the training happens here

log_reg = pipeline.named_steps["model"]
print(f"Trained: {log_reg.coef_.shape[1]} weights + 1 intercept (b = {log_reg.intercept_[0]:.2f}), "
      f"converged in {log_reg.n_iter_[0]} iterations")

### 8.1 What the intern learned
Each bar is a weight. Red adds points (raises the chance), grey removes points.

In [ ]:
def readable_names(preprocess):
    names = []
    for name, step, cols in preprocess.transformers_:
        if name == "scale":
            names += [f"{c} (+1 std. dev.)" for c in cols]
        elif name == "boxes":
            for c, edges in zip(cols, step.bin_edges_):
                names += [f"{c} {edges[i]:.3g}–{edges[i + 1]:.3g}" for i in range(len(edges) - 1)]
        elif name == "binary":
            names += list(cols)
        elif name == "onehot":
            for c, cats in zip(cols, step.categories_):
                names += [f"{c} = {k}" for k in cats]
    return names

names = readable_names(pipeline.named_steps["preprocess"])
assert len(names) == log_reg.coef_.shape[1]
weights = pd.Series(log_reg.coef_[0], index=names).sort_values()

fig, ax = plt.subplots(figsize=(9, max(5, len(weights) * 0.28)))
ax.barh(weights.index, weights.values, color=[RED if v > 0 else GREY for v in weights.values])
ax.axvline(0, color=INK, lw=1)
ax.set_title("Learned weights"); ax.set_xlabel("← lowers the chance   |   raises the chance →")
plt.tight_layout(); plt.show()

print("Strongest positive:", ", ".join(f"{n} ({w:+.2f})" for n, w in weights[::-1].head(5).items()))
print("Strongest negative:", ", ".join(f"{n} ({w:+.2f})" for n, w in weights.head(5).items()))

**How to read the weights.** These are the rules the intern found in real videos; nobody wrote them in advance. Two cautions: with a few hundred videos, small weights (close to 0) can be noise; and a weight shows an *association*, not a cause (see Limitations).

<a id="s9"></a>
## Step 9 — Evaluate: accuracy and confusion matrix

> **Intern analogy.** Time to open the drawer. The intern guesses each hidden video and we grade the answers. To know whether the grade is actually good, we compare it with the **lazy intern** who always answers "not viral".

In [ ]:
pred = pipeline.predict(X_test)
proba = pipeline.predict_proba(X_test)[:, 1]

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
accuracy = accuracy_score(y_test, pred)
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
precision = precision_score(y_test, pred, zero_division=0)
recall, auc = recall_score(y_test, pred), roc_auc_score(y_test, proba)

print(f"Accuracy of the model:                {accuracy:.1%}  ({(pred == y_test).sum()} of {len(y_test)} correct)")
print(f"Accuracy of 'always says not viral':  {baseline_accuracy:.1%}")
print(f"Precision: {precision:.1%}   Recall: {recall:.1%}   AUC: {auc:.3f}")

In [ ]:
matrix = confusion_matrix(y_test, pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(matrix, display_labels=["not viral", "viral"]).plot(ax=ax, cmap="Oranges", colorbar=False)
ax.set_xlabel("predicted by the model"); ax.set_ylabel("what really happened"); ax.set_title("Confusion matrix (test set)")
plt.show()

(tn, fp), (fn, tp) = matrix
print(f"True negatives:  {tn:4d}  not viral, and the model said not viral")
print(f"False positives: {fp:4d}  false alarm")
print(f"False negatives: {fn:4d}  viral, but the model missed it")
print(f"True positives:  {tp:4d}  viral, and the model caught it")

| Metric | Question it answers |
|---|---|
| **Accuracy** | out of all test videos, what % did the model get right? |
| **Precision** | when the model says "viral", how often is it right? |
| **Recall** | out of the videos that really went viral, how many did the model catch? |
| **AUC** | given one viral and one non-viral video at random, how often does the model score the viral one higher? (0.5 = coin flip, 1.0 = perfect) |

**Reading the result.** Because 25% of videos are viral by definition, the lazy intern already scores ~75% accuracy, so accuracy alone is a weak measure here. Compare **precision** with the 25% you would get by guessing, and **AUC** with the 0.5 of a coin flip. Real virality depends on many things that are not in the checklist (the idea itself, trends, the platform's algorithm), so a modest score is expected and honest.

### 9.1 The decision threshold
> **Intern analogy.** The intern gives a percentage, and *we* decide from which percentage they should shout "viral!". A cautious intern (50%) rarely raises false alarms but misses many hits. A bolder one (30%) catches more viral videos, with more false alarms.

In [ ]:
rows = []
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    p = (proba >= threshold).astype(int)
    rows.append({"threshold": f"{threshold:.0%}", "accuracy": accuracy_score(y_test, p),
                 "precision": precision_score(y_test, p, zero_division=0), "recall": recall_score(y_test, p)})
pd.DataFrame(rows).set_index("threshold").style.format("{:.1%}")

### 9.2 Was it luck? Cross-validation
> **Intern analogy.** One exam could have been unusually easy or hard, especially a small one. So we run 5 mini-exams, each time hiding a different fifth of the study pile, and check that the grades are consistent.

In [ ]:
final_model = build_pipeline(LogisticRegression(max_iter=1000), N_BOXES)
cv_accuracy = cross_val_score(final_model, X_train, y_train, cv=cv, scoring="accuracy")
cv_auc = cross_val_score(final_model, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"5-fold cross-validation accuracy: {cv_accuracy.mean():.1%} ± {cv_accuracy.std():.1%}   {np.round(cv_accuracy, 3)}")
print(f"5-fold cross-validation AUC:      {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

<a id="s10"></a>
## Step 10 — Test with completely new data

> **Intern analogy.** The real job: videos whose outcome **nobody knows yet**. There is no answer to check, only the intern's prediction. This is what the model is for.

`new_posts.json` holds real Shorts that were less than 3 days old when collected. Their final views are not known yet, so they are not in the dataset and have no label. Check them again in a week to see whether the predictions came true.

In [ ]:
new_posts = []
if os.path.exists("new_posts.json"):
    with open("new_posts.json", encoding="utf-8") as f:
        new_posts = json.load(f)

if not new_posts:
    print("new_posts.json is empty or missing: run collect_real_videos.ipynb to collect recent videos.")
    result = None
else:
    X_new = pd.DataFrame([px.extract(p, USE_LLM) for p in new_posts])[FEATURES]
    result = X_new.assign(chance_viral=pipeline.predict_proba(X_new)[:, 1])
    result["prediction"] = np.where(result.chance_viral >= 0.5, "VIRAL", "not viral")
    result.insert(0, "title", [p["title"][:45] for p in new_posts])
    result.insert(1, "url", [p["url"] for p in new_posts])
result.style.format({"chance_viral": "{:.1%}"}) if result is not None else None

<a id="s11"></a>
## Step 11 — Export the trained model as `.pkl`

> **Intern analogy.** Training took effort, and you do not want the intern to study from scratch every time. So you write down everything they learned (every weight, every box edge, every category) in a **notebook they can take anywhere**. That notebook is the `.pkl` file.

**Technical note.** `pickle` turns the whole `Pipeline` (preprocessing + weights) into a file. It contains only standard scikit-learn components, so opening it needs nothing but scikit-learn. Metadata is saved with it, including the scikit-learn version, because a `.pkl` should be opened with the same version that created it.

In [ ]:
bundle = {
    "name": "Will It Go Viral? — Logistic Regression (real YouTube Shorts)",
    "author": "Gustavo Pugliezi",
    "pipeline": pipeline,
    "features": FEATURES,
    "threshold": 0.5,
    "viral_cutoff_views_per_subscriber": float(VIRAL_CUTOFF),
    "trained_on": len(X_train),
    "test_accuracy": float(accuracy),
    "sklearn_version": sklearn.__version__,
}
with open("viral_model.pkl", "wb") as f:
    pickle.dump(bundle, f)
print(f"Saved viral_model.pkl ({os.path.getsize('viral_model.pkl') / 1024:.1f} KB)")

<a id="s12"></a>
## Step 12 — Open and run in any application

> **Intern analogy.** Now anyone can borrow the intern's notebook and make the same predictions, without any studying. We prove it twice: first by reopening the file from scratch, then with a completely separate program.

### 12.1 Reopening the `.pkl` from scratch

In [ ]:
with open("viral_model.pkl", "rb") as f:
    loaded = pickle.load(f)

assert np.allclose(loaded["pipeline"].predict_proba(X_test)[:, 1], proba)
print(f"{loaded['name']} | by {loaded['author']} | test accuracy {loaded['test_accuracy']:.1%}")
print("Predictions are identical to the original model ✔")

### 12.2 A separate command-line application
`predict_app.py` is an independent program: it reads a JSON file of posts (in the collector's format), extracts the features with `post_extractor.py` and predicts with `viral_model.pkl`. It runs from any terminal:
```bash
python predict_app.py new_posts.json
```

In [ ]:
%%writefile predict_app.py
"""Predicts whether YouTube Shorts will go viral, using viral_model.pkl.

Usage:  python predict_app.py posts.json [--llm]
"""
import json
import pickle
import sys
import warnings
from pathlib import Path

import pandas as pd
import sklearn

import post_extractor as px

HERE = Path(__file__).resolve().parent


def load_model(path=HERE / "viral_model.pkl"):
    with open(path, "rb") as f:
        bundle = pickle.load(f)
    if bundle["sklearn_version"] != sklearn.__version__:
        warnings.warn(f"model saved with scikit-learn {bundle['sklearn_version']}, running {sklearn.__version__}")
    return bundle


def predict(posts, bundle, use_llm=False):
    X = pd.DataFrame([px.extract(p, use_llm) for p in posts])[bundle["features"]]
    chances = bundle["pipeline"].predict_proba(X)[:, 1]
    return [(p.get("title", p.get("id", "?")), float(c), c >= bundle["threshold"]) for p, c in zip(posts, chances)]


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit(__doc__)
    bundle = load_model()
    posts = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
    print(f"{bundle['name']} (trained on {bundle['trained_on']} videos)\n")
    for title, chance, viral in predict(posts, bundle, use_llm="--llm" in sys.argv):
        print(f"{title[:40]:40s} {chance:6.1%}  {'█' * round(chance * 20):20s}  {'VIRAL' if viral else 'not viral'}")

In [ ]:
if new_posts:
    !"{sys.executable}" predict_app.py new_posts.json
else:
    print("No new posts to predict.")

---
## Summary

In [ ]:
summary = pd.Series({
    "Real Shorts collected": f"{len(posts):,} from {len(channels):,} channels",
    "Features per video": len(FEATURES),
    "Viral definition": f"top {VIRAL_SHARE:.0%}: ≥ {VIRAL_CUTOFF:.2f} views per subscriber",
    "Training / testing videos": f"{len(X_train):,} / {len(X_test):,}",
    "Accuracy (test)": f"{accuracy:.1%}",
    "Accuracy of 'always not viral'": f"{baseline_accuracy:.1%}",
    "Precision / Recall": f"{precision:.1%} / {recall:.1%}",
    "AUC (test)": f"{auc:.3f}",
    "Cross-validation accuracy / AUC": f"{cv_accuracy.mean():.1%} / {cv_auc.mean():.3f}",
    "Exported model": "viral_model.pkl",
}, name="")
summary.to_frame()

### Quality checks built into this notebook
| Risk | How it is handled |
|---|---|
| **Answer leaking into the features** | extractors only see `VISIBLE_FIELDS`; views, likes and subscribers are used for the label only |
| **Unfinished videos** | only Shorts at least 7 days old are used for training and testing |
| **Peeking at the exam** | exploration (7.1) uses the training set only; box edges, scaler and categories are learned inside `fit` on training data; the number of boxes is chosen by cross-validation on training data |
| **Silent bad data** | `validate_features` rejects unknown categories, missing values and impossible numbers |
| **Lucky split** | 5-fold cross-validation |
| **Broken export** | the reloaded `.pkl` is checked to give identical predictions; the app warns about version mismatches |
| **Reproducibility** | fixed random seed; the raw data is saved in `posts.json` |

### Limitations
1. **Sample bias.** Videos come from YouTube search, which favours popular videos, and only from Portuguese-language searches in 6 topics. The model describes *this* sample, not all of YouTube.
2. **Automatic measurements can be wrong.** Face detection misses faces in profile or in poor light; the cut counter can mistake a fast camera move for a cut; the loop check compares colours, not content; the classroom rules for hooks only recognise certain phrases. A sample of the measurements should be checked by watching the videos.
3. **Correlation is not causation.** A positive weight means viral videos *tend to* have that feature, not that adding it *causes* virality.
4. **Missing factors.** The idea itself, trends, thumbnails, music and the platform's algorithm are not in the checklist, which is why a large part of the outcome cannot be predicted.
5. **Small data.** A few hundred videos give a noisy test score; the cross-validation result is the more reliable estimate.